In [1]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.


BOXBOX - F1 Strategy Intelligence Dashboard

Phase 3 :- Degradation Modelling

Models to train:
1. Polynomial Regression
2. Ridge Regression
3. Random Forest

SHAP Interpretability

In [2]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
import joblib
import shap

from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

warnings.filterwarnings('ignore')

SETUP

In [3]:
BASE = r'C:\Users\adity\Desktop\BoxBox'

os.makedirs(os.path.join(BASE, 'models'),exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'processed'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'outputs'),exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase3_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

FEATURES USED FOR MODELING

These are the input features every model will use

TyreLife is the core variable - the others give context that helps Ridge and Random Forest capture real-world effects beyound just tire age.


In [4]:
FEATURES = ['TyreLife', 'TrackTemp', 'StintProgress', 'FuelCorrectedLapTime']
TARGET = 'LapTimeSeconds'

MIN_SAMPLES = 15 #Minimum laps required to train models for a group

LOAD ENGINEERED DATA

In [5]:
def load_data():
    path = os.path.join(BASE, 'data', 'processed', 'engineered_laps.csv')
    df = pd.read_csv(path)
    log.info(f"Loaded engineered_laps.csv: {len(df)} rows")

    '''Drop rows with missing values in our feature set - 
    models can't handle NaN inputs'''

    before = len(df)
    df = df.dropna(subset=FEATURES + [TARGET])
    log.info(f"After dropping missing values: {len(df)}" f"(removed {before - len(df)})")

    return df

POLYNOMIAL REGRESSION TRAINING

Fits a curved line using only TyreLife - captures the "cliff" shape without needing multiple features.

Degree = 2 means it can fit a U-shaped or accelerating curve

In [6]:
def train_polynomial(X, y):
    model = make_pipeline(
        PolynomialFeatures(degree = 2, include_bias = False),
        Ridge(alpha = 1.0)
    )

    X_single = X[['TyreLife']].values

    kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
    scores = cross_val_score(
        model, X_single, y, cv=kf,
        scoring = 'neg_root_mean_squared_error'
    )
    rmse = -scores.mean()

    model.fit(X_single, y)
    return model, rmse

TRAIN RIDGE REGRESSION

Linear regression across ALL features with L2 regularization to prevent overfitting when features are correlated

(e.g. StintProgress and TyreLife move togther)

In [7]:
def train_ridge(X, y):
    model = Ridge(alpha=1.0)

    X_full = X[FEATURES].values

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_full, y, cv=kf,
        scoring='neg_root_mean_squared_error'
    )
    rmse = -scores.mean()

    model.fit(X_full, y)
    return model, rmse

TRAIN RANDOM FOREST

Ensemble of decison trees - captures non-linear interactions between all features automatically, n_estimators=100 trees, max_depth limited to avoid overfitting on small per-circuit sample sizes.

In [8]:
def train_random_forest(X, y):
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=6, 
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    X_full = X[FEATURES].values

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_full, y, cv=kf,
        scoring='neg_root_mean_squared_error'
    )
    rmse = -scores.mean()

    model.fit(X_full, y)
    return model, rmse

EXTRACT DEGRADATION RATE FROM SELECTED MODEL

Regardless of which model won, we calculate: if TyreLife increases by 1 lap (all else held at its average), how much does predicted lap time change? 

This is done by predicting at two tire-age points and taking the difference - works identically for all three model types since we just call .predict() twice.

In [9]:
def calculate_degradation_rate(model, model_type, X, avg_tyre_life):
    if model_type  == 'Polynomial':
        base = model.predict([[avg_tyre_life]])[0]
        plus_one = model.predict([[avg_tyre_life + 1]])[0]
    else:
        '''Ridge and Random Forest need full features now
        we hold every other feature as it's group average
        and only change TyreLife'''
        avg_row = X[FEATURES].mean()

        base_row = avg_row.copy()
        base_row['TyreLife'] = avg_tyre_life
        plus_row = avg_row.copy()
        plus_row['TyreLife'] = avg_tyre_life + 1

        base = model.predict([base_row.values])[0]
        plus_one = model.predict([plus_row.values])[0]

    return round(plus_one - base, 4)

MAIN MODELLING LOOP

Loop through every circuit + compound combination -> trains all models -> picks the best -> extracts degradation rate -> stores everything

In [10]:
def run_degradation_modeling(df):
    log.info("TRAINING DEGRADATION MODELS")
    log.info("-" * 60)

    results = []
    saved_models = {}
    metric_rows = []

    groups = df.groupby(['CircuitName', 'Compound'])
    total_groups = len(groups)
    log.info(f"Total circuit + compound groups to process: {total_groups}")

    for (circuit, compound), group in groups:
        if len(group) < MIN_SAMPLES:
            log.warning(f" Skipping {circuit} / {compound} - "
                        f" only {len(group)} samples (need {MIN_SAMPLES}+)")
            continue

        X = group[FEATURES].copy()
        y = group[TARGET].copy()
        avg_tyre_life = X['TyreLife'].mean()

        #Train all three models
        poly_model, poly_rmse = train_polynomial(X, y)
        ridge_model, ridge_rmse = train_ridge(X, y)
        rf_model, rf_rmse = train_random_forest(X, y)

        #Pick the best model based on lowest RMSE
        candidates = {
            'Polynomial': (poly_model, poly_rmse),
            'Ridge': (ridge_model, ridge_rmse),
            'RandomForest': (rf_model, rf_rmse)
        }
        best_type = min(candidates, key=lambda k: candidates[k][1])
        best_model, best_rmse = candidates[best_type]

        #Extract degradation rate from the WINNING model
        deg_rate = calculate_degradation_rate(
            best_model, best_type, X, avg_tyre_life
        )

        log.info(f" {circuit} / {compound}: winner={best_type}"
                f"(RMSE={best_rmse:.3f}) | deg_rate={deg_rate:+.4f} s/lap")
        
        #save results row
        results.append({
            'CircuitName': circuit,
            'Compound': compound,
            'SelectedModel': best_type,
            'DegradationRate': deg_rate,
            'RMSE': round(best_rmse, 4),
            'SampleSize': len(group),
            'AvgTyreLife': round(avg_tyre_life, 1),
            'AvgLapTime': round(y.mean(), 3)
        })

        #Save metrics for ALL three models (for comparison later)
        for model_type, (_, rmse) in candidates.items():
            metric_rows.append({
                'CircuitName': circuit,
                'Compound': compound,
                'ModelType': model_type,
                'RMSE': round(rmse, 4),
                'Selected': model_type == best_type
            })

        #Store all three trained models for this group
        key = f"{circuit} | {compound}"
        saved_models[key] = {
            'Polynomial': poly_model,
            'Ridge': ridge_model,
            'RandomForest': rf_model,
            'selected': best_type
        }

    degradation_df = pd.DataFrame(results)
    metrics_df = pd.DataFrame(metric_rows)

    return degradation_df, metrics_df, saved_models

SHAP INTERPRETABILITY

Runs SHAP on the Random Forest model for each circuit across all compounds combined to explain which features drive lap time predictions most. We use TreeExplainer since it's fast and exact for tree-based models like RandomForest

In [11]:
def compute_shap_values(df):
    log.info("COMPUTING SHAP VALUES")
    log.info("-" * 60)

    shap_rows = []

    for circuit, group in df.groupby('CircuitName'):
        if len(group) < MIN_SAMPLES:
            continue

        X = group[FEATURES].copy()
        y = group[TARGET].copy()

        '''Train a Random Forest specifically for SHAP analysis
        (using all compounds together at this circuit, since
        we want overall feature importance for the circuit)'''

        rf = RandomForestRegressor(
            n_estimators=100, max_depth=6,
            min_samples_leaf=3, random_state = 42, n_jobs=-1
        )
        rf.fit(X, y)
        
        explainer = shap.TreeExplainer(rf)
        shap_values = explainer.shap_values(X)

        #Average absolute SHAP value per feature = overall importance
        mean_abs_shap = np.abs(shap_values).mean(axis=0)

        for feature, importance in zip(FEATURES, mean_abs_shap):
            shap_rows.append({
                'CircuitName': circuit,
                'Feature': feature,
                'MeanAbsSHAP': round(importance, 4)
            })
        
        log.info(f" {circuit}: SHAP computed"
                f"({len(group)} laps, {len(FEATURES)} features)")
    
    shap_df = pd.DataFrame(shap_rows)
    return shap_df

MAIN PIPELINE

In [12]:
def main():
    log.info("BOXBOX PHASE 3: DEGRADATION MODELING")
    log.info("-" * 50)

    df = load_data()

    degradation_df, metrics_df, saved_models = run_degradation_modeling(df)
    shap_df = compute_shap_values(df)

    #Save degradation profiles
    deg_path = os.path.join(BASE, 'data', 'processed', 'degradation_profiles.csv')
    degradation_df.to_csv(deg_path, index=False)
    log.info(f"Saved: {deg_path} ({len(deg_path)} rows)")

    #Save model comparison metrics
    metrics_path = os.path.join(BASE, 'data', 'outputs', 'model_metrics.csv')
    metrics_df.to_csv(metrics_path, index=False)
    log.info(f"Saved: {metrics_path} ({len(metrics_path)} rows)")

    #Save SHAP values
    shap_path = os.path.join(BASE, 'data', 'outputs', 'shap_values.csv')
    shap_df.to_csv(shap_path, index = False)
    log.info(f"Saved: {shap_path} ({len(shap_df)} rows)")

    #Save trained models
    models_path = os.path.join(BASE, 'models', 'degradation_models.pkl')
    joblib.dump(saved_models, models_path)
    log.info(f"Saved: {models_path} ({len(saved_models)} circuit+compound models)")

    #Summary
    log.info("Phase 3 Summary")
    log.info("-" * 50)

    log.info(f"\nmodel selection breakdown:")
    log.info(degradation_df['SelectedModel'].value_counts().to_string())

    log.info(f"\nAverage RMSE by model type (across all groups):")
    log.info(metrics_df.groupby('ModelType')['RMSE'].mean().round(3).to_string())

if __name__ == '__main__':
    main()

2026-07-28 07:17:14,176 - INFO - BOXBOX PHASE 3: DEGRADATION MODELING
2026-07-28 07:17:14,178 - INFO - --------------------------------------------------
2026-07-28 07:17:14,278 - INFO - Loaded engineered_laps.csv: 20887 rows
2026-07-28 07:17:14,285 - INFO - After dropping missing values: 20887(removed 0)
2026-07-28 07:17:14,286 - INFO - TRAINING DEGRADATION MODELS
2026-07-28 07:17:14,287 - INFO - ------------------------------------------------------------
2026-07-28 07:17:14,296 - INFO - Total circuit + compound groups to process: 66
2026-07-28 07:17:15,383 - INFO -  Abu Dhabi / HARD: winner=RandomForest(RMSE=0.335) | deg_rate=-0.0030 s/lap
2026-07-28 07:17:16,414 - INFO -  Abu Dhabi / MEDIUM: winner=RandomForest(RMSE=0.297) | deg_rate=-0.0359 s/lap
2026-07-28 07:17:17,131 - INFO -  Abu Dhabi / SOFT: winner=Ridge(RMSE=0.202) | deg_rate=-0.0087 s/lap
2026-07-28 07:17:17,897 - INFO -  Australia / HARD: winner=RandomForest(RMSE=0.472) | deg_rate=+0.0117 s/lap
2026-07-28 07:17:18,841 - I